## Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [2]:
# Setup the model
import os

from langchain.chat_models import init_chat_model
from dotenv import load_dotenv

load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")

In [4]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    name: str = Field(description="This is the name of the movie")
    year: int = Field(description="This is the year the movie was released")
    director: str = Field(description="This is the name of the movie director")
    rating: float = Field(description="This is the movie rating out of 10")

model_with_structure = model.with_structured_output(Movie)

In [3]:
response = model.invoke("Give me details about the movie Inception")
response.content

'<think>\nOkay, so I need to figure out details about the movie Inception. Let me start by recalling what I know. Inception is a 2010 film directed by Christopher Nolan. The lead actor is Leonardo DiCaprio, right? The movie is about dreams within dreams or something like that. I think it\'s a sci-fi action film. The plot involves entering people\'s dreams to plant ideas or extract them, maybe? There\'s a concept called "inception," which is about planting an idea into someone\'s subconscious. The director is known for complex narratives, like in Memento or The Dark Knight. The movie has a lot of visual effects, especially with the folding city or the rotating hallway fight scene. The cast includes Tom Hardy, Ellen Page, and others. The music was by Hans Zimmer, which is a big part of the film\'s atmosphere. There\'s a lot of philosophical underpinnings about reality vs. dreams. The ending is open-ended, leaving the audience to wonder if it\'s a dream or not. I should also mention the t

In [10]:
response = model_with_structure.invoke("Give me details about the movie Inception")
response

Movie(name='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [ ]:
## Message Output (raw) along with the parsed structure
model_with_structure_raw = model.with_structured_output(Movie, include_raw=True)
response = model_with_structure_raw.invoke("Give me details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check what tools I have available. There\'s a Movie function that requires name, year, director, and rating. I need to make sure I have all that information for Inception.\n\nFirst, the name is obviously "Inception". The director is Christopher Nolan. I remember he directed it. The year it was released was 2010. As for the rating, I think it\'s around 8.8 on IMDb. Let me confirm that. Yes, IMDb gives it 8.8/10. So all the required parameters are there. I can now call the Movie function with these details. I just need to structure the JSON correctly with the parameters in the right format. Make sure the rating is a number and the year is an integer. No typos in the director\'s name. Alright, that should cover it.\n', 'tool_calls': [{'id': 'vbb13a6e7', 'function': {'arguments': '{"director":"Christopher Nolan","name":"Inception","rating":8.8,"year"

In [6]:
## Nested models
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    name: str = Field(description="This is the name of the movie")
    year: int = Field(description="This is the year the movie was released")
    cast: list[Actor]
    genre: list[str]
    director: str = Field(description="This is the name of the movie director")
    rating: float = Field(description="This is the movie rating out of 10")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Give me details about the movie Inception")
response

MovieDetails(name='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Joker')], genre=['Action', 'Sci-Fi', 'Thriller'], director='Christopher Nolan', rating=8.8)

In [ ]:
model.profile
# model_with_structure.profile  # Does not work with structured output model

In [20]:
## Using Response format with agents
import os
from pydantic import BaseModel, Field
from langchain.agents import create_agent

load_dotenv()
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

agent = create_agent(
    model="gpt-5",
    response_format=MovieDetails
)
messages = {
    "messages": [{
        "role" : "user",
        "content": "Give me details about the Movie inception"
    }]
}

result = agent.invoke(messages)
print(result)

{'messages': [HumanMessage(content='Give me details about the Movie inception', additional_kwargs={}, response_metadata={}, id='703860d3-ea9e-4dcc-a60b-151238075db4'), AIMessage(content='{"name":"Inception","year":2010,"cast":[{"name":"Leonardo DiCaprio","role":"Dom Cobb"},{"name":"Joseph Gordon-Levitt","role":"Arthur"},{"name":"Elliot Page","role":"Ariadne"},{"name":"Tom Hardy","role":"Eames"},{"name":"Ken Watanabe","role":"Saito"},{"name":"Dileep Rao","role":"Yusuf"},{"name":"Cillian Murphy","role":"Robert Fischer"},{"name":"Marion Cotillard","role":"Mal"},{"name":"Michael Caine","role":"Professor Stephen Miles"},{"name":"Tom Berenger","role":"Peter Browning"},{"name":"Pete Postlethwaite","role":"Maurice Fischer"}],"genre":["Science fiction","Action","Thriller","Heist"],"director":"Christopher Nolan","rating":8.8}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1978, 'prompt_tokens': 292, 'total_tokens': 2270, 'completion